In [23]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [24]:
df = pd.read_csv("D:\\Project\\MLOP\\artifacts\\data_ingestion\\WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

In [25]:
df.info()

In [26]:
df.describe()

In [27]:
df.nunique()

# Visualize

In [33]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
sns.countplot(data=df, x='InternetService',hue='Churn')

plt.subplot(1, 2, 2)
sns.countplot(data=df, x='Contract',hue='Churn')
plt.show()

A. Ordinal Encoding cho Contract
- Vì chúng ta thấy rõ xu hướng: Hợp đồng càng dài hạn $\rightarrow$ Tỉ lệ rời bỏ càng thấp. Việc đặt giá trị theo thứ tự $0, 1, 2$ sẽ giúp mô hình (đặc biệt là các thuật toán dựa trên cây như Random Forest hay XGBoost) hiểu được mối quan hệ tuyến tính này.

B. One-Hot Encoding cho InternetService
- Vì Fiber optic có đặc tính rời bỏ rất khác biệt so với DSL và No, chúng ta nên tách chúng ra thành các cột riêng biệt để mô hình không bị nhầm lẫn về mặt định lượng.

In [34]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='PaymentMethod', hue='Churn')
plt.show()

Electronic check : Cột màu cam (Churn) của phương thức này cao khủng khiếp so với 3 phương thức còn lại. Số lượng người rời bỏ khi dùng Electronic check gần bằng số người ở lại.

Nhóm tự động (Automatic) rất trung thành: Cả Bank transfer và Credit card đều có tỉ lệ rời bỏ rất thấp.

Kết luận: Phương thức thanh toán không chỉ là một cái tên, nó là một đặc trưng (feature) mạnh để dự đoán khách hàng có rời đi hay không.

In [28]:
cols_3_values = [col for col in df.columns if df[col].nunique() == 3]

file_path = "cols_3_values.txt"

with open(file_path, "w", encoding="utf-8") as f:
    for col in cols_3_values:
        f.write(f"\n🔹 Column: {col}\n")
        f.write(str(df[col].value_counts()))
        f.write("\n" + "-"*40 + "\n")

In [30]:
from transform_data import Transform
transformer = Transform()

# 1. Xử lý các cột Binary (Yes/No)
bin_features = ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]
df = transformer.bin_col(df, bin_features) # Mặc định No=0, Yes=1

# 2. Xử lý riêng cột gender (Female/Male)
df = transformer.bin_col(df, 'gender', zero_val='Female', one_val='Male')

# 3. Xử lý các cột có 3 giá trị (No, Yes, No service)
triple_features = [
    "MultipleLines", "OnlineSecurity", "OnlineBackup", 
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
]
df = transformer.reduce_triple_values(df, triple_features)

In [31]:
df.info()

In [32]:
df.nunique()

In [35]:
# A. Ordinal Encoding cho Contract (Dựa trên thứ tự cam kết)
contract_map = {
    "Month-to-month": 0, 
    "One year": 1, 
    "Two year": 2
}
df = transformer.encode_ordinal(df, 'Contract', contract_map)

# B. One-Hot Encoding cho InternetService
df = transformer.encode_onehot(df, 'InternetService', 'IS')

# C. One-Hot Encoding cho PaymentMethod (Dựa trên phân tích Electronic Check rất đặc thù của bạn)
df = transformer.encode_onehot(df, 'PaymentMethod', 'PM')

# D. Ép kiểu TotalCharges về số và xử lý Null
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = transformer.fill_null(df, 'TotalCharges', strategy='constant', fill_value=0)

# E. Loại bỏ cột customerID (Không có giá trị dự báo)
df = transformer.drop_columns(df, 'customerID')

### Engagement Feature Engineering
Tạo đặc trưng `TotalServices` bằng cách cộng dồn các dịch vụ mà khách hàng đang sử dụng. 

In [ ]:
addon_cols = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 
    'OnlineBackup', 'DeviceProtection', 'TechSupport', 
    'StreamingTV', 'StreamingMovies'
]
df = transformer.create_engagement_features(df, addon_cols)
df[['TotalServices']].head()

### Correlation Matrix
Sau khi đã Encode các biến phân loại, chúng ta xem xét ma trận tương quan để thấy mối quan hệ giữa các biến với `Churn`.

In [ ]:
plt.figure(figsize=(16, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Correlation Matrix of Features")
plt.show()

print("Correlation with Churn:")
print(corr['Churn'].sort_values(ascending=False))

### Handling Class Imbalance (SMOTE)
Dataset này thường bị mất cân bằng (số người không rời bỏ nhiều hơn số người rời bỏ). Chúng ta sử dụng SMOTE để cân bằng dữ liệu trước khi huấn luyện.

In [ ]:
from imblearn.over_sampling import SMOTE

X = df.drop('Churn', axis=1)
y = df['Churn']

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

print("Trước khi SMOTE:", y.value_counts().to_dict())
print("Sau khi SMOTE:", y_res.value_counts().to_dict())

df_balanced = pd.concat([X_res, y_res], axis=1)